# NB_lab_test_trends — Gold Layer

Builds `adb_caresync.gold.lab_test_trends`: one row per test type per calendar month with abnormal/critical rates and month-over-month trend via LAG window. Full rebuild every run.

In [0]:
%sql
-- ============================================================
-- gold.lab_test_trends
-- Grain : one row per test_name per calendar month
-- Load  : FULL — CREATE OR REPLACE TABLE ... AS SELECT
-- Sources: silver.lab_results (driving, grouped by test_name + year + month)
--          silver.calendar    (LEFT JOIN on date_key for month/quarter names)
-- Window : LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month)
-- ============================================================

CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS

WITH monthly_agg AS (
  SELECT
    test_name,
    YEAR(test_date)    AS year,
    MONTH(test_date)   AS month,
    -- First day of the month — used to join to calendar
    DATE_TRUNC('MONTH', MIN(test_date))                                    AS month_start_date,
    COUNT(*)                                                               AS total_tests,
    COUNT(*) FILTER (WHERE result_status = 'Abnormal')                     AS abnormal_test_count,
    COUNT(*) FILTER (WHERE result_status = 'Critical')                     AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),

enriched AS (
  SELECT
    m.test_name,
    m.year,
    m.month,
    cal.month_name,
    cal.quarter_name,
    m.total_tests,
    m.abnormal_test_count,
    m.critical_test_count,
    ROUND(CAST(m.abnormal_test_count AS DOUBLE) / m.total_tests * 100, 2)  AS abnormal_rate_pct,
    ROUND(CAST(m.critical_test_count AS DOUBLE) / m.total_tests * 100, 2)  AS critical_rate_pct
  FROM monthly_agg m
  LEFT JOIN adb_caresync.silver.calendar cal
    ON CAST(date_format(m.month_start_date, 'yyyyMMdd') AS INT) = cal.date_key
),

with_lag AS (
  SELECT
    *,
    LAG(abnormal_rate_pct) OVER (
      PARTITION BY test_name
      ORDER BY year, month
    ) AS prev_month_abnormal_rate_pct
  FROM enriched
)

SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  CAST(abnormal_rate_pct             AS DECIMAL(5, 2))  AS abnormal_rate_pct,
  CAST(critical_rate_pct             AS DECIMAL(5, 2))  AS critical_rate_pct,
  CAST(prev_month_abnormal_rate_pct  AS DECIMAL(5, 2))  AS prev_month_abnormal_rate_pct,
  CAST(
    CASE
      WHEN prev_month_abnormal_rate_pct IS NULL THEN NULL
      ELSE ROUND(abnormal_rate_pct - prev_month_abnormal_rate_pct, 2)
    END
  AS DECIMAL(6, 2))                                     AS abnormal_rate_change_pct

FROM with_lag

In [0]:
record_count = spark.table("adb_caresync.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))